# Sources and candidate seed

Purpose: trace pinned source files through validation, immutable Parquet, and a resumable checkpoint. Defaults use invented fixtures only. Use the Baseline Python 3.12 kernel. Expected runtime is seconds; temporary fixture artifacts stay under `.cache/`. No NBA request, database access, or forecast publication occurs.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pyarrow.parquet as pq

from hoop.domains.nba.sources.seed import build_seed
from hoop.platform.local_artifacts import LocalArtifacts

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve()
source = ROOT / "tests/fixtures/nba/seed"
manifest = json.loads((source / "manifest.json").read_text())
inventory = json.loads((source / "inventory.json").read_text())
print("Evidence:", manifest["evidence_kind"], "| dataset version:", manifest["dataset_version"])

## Run and inspect

The CLI and notebook call the same tested function. Every source hash is checked. Null minutes, DNP evidence, and unknown participation remain distinct. The fixture has source calendar dates, so no UTC start time is invented.

In [ ]:
cache = ROOT / ".cache"
cache.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(dir=cache) as temp:
    archive = Path(temp)
    seed_report = build_seed(source, manifest, archive, inventory=inventory)
    player_path = LocalArtifacts(archive).verify(seed_report["normalized"]["players"])
    display(
        pq.read_table(player_path).to_pandas()[
            ["game_id", "player_id", "seconds", "points", "participation"]
        ]
    )
    replay_report = build_seed(source, manifest, archive, inventory=inventory)
    print("Rows:", seed_report["row_counts"], "| replay reused:", replay_report["reused"])
    print(
        "Status:", seed_report["seed_status"], "| source accepted:", seed_report["source_accepted"]
    )

    from hoop.domains.nba.history.raw import stage_raw_archives
    from hoop.domains.nba.history.seed import plan_seed
    from hoop.domains.nba.sources.seed_audit import audit_quarantine

    audit = audit_quarantine(seed_report, archive)
    print("Quarantine audit:", audit["quarantine_count"], "| repairs:", audit["repairs_applied"])
    print(audit["limitation"])

    # Invented fixture approval and request times are for this rehearsal only.
    fixture_approval = {
        "kind": "synthetic_fixture",
        "manifest_sha256": seed_report["manifest_artifact"]["sha256"],
        "parser_version": seed_report["parser_version"],
        "calendar_convention": "source_calendar_verified",
    }
    fixture_observations = {
        name: {
            "requested_at": "2026-10-24T11:59:00Z",
            "received_at": manifest["collected_at"],
            "sha256": sha,
        }
        for name, sha in manifest["files"].items()
    }
    staged_raw = stage_raw_archives(seed_report, archive)
    print("Encoded raw bytes:", sum(a["encoded_byte_count"] for a in staged_raw["files"].values()))
    import_plan = plan_seed(
        seed_report, archive, manifest, fixture_approval, fixture_observations, staged_raw
    )
    print("Import plan only:", import_plan["counts"], "| blockers:", import_plan["blockers"])

## Interpretation and full-data operation

A replay verifies immutable artifacts and checkpoint identity before reuse. Interruptions cannot turn a partial Parquet file into a completed seed. Raw data and repair overlays retain separate provenance. The archive can report incomplete coverage while producing inspectable valid records.

This fixture does not prove NBA access, complete history, or historical pregame rosters. Use the explicit CLI command in `docs/seed-backfill.md` for retained real files. Source/seed acceptance, backup verification, and production activation are separate gates.

DB-003 adds a dry selection plan. The fixture approval and request times are invented test inputs, not owner approval or evidence of a live request. Production import is a separate gated CLI operation; Run All does not open a database or upload data.

The read-only quarantine audit binds each flagged row to original bytes and shows overlapping normalized records. It never converts zeros into DNP evidence, repairs a row, or accepts a seed. See `docs/seed-backfill.md` for the explicit real-archive audit command.